In [ ]:
# Setup (added for the release): the notebook's folder on sys.path, then work in its data folder.
# The curation ran with the current directory = that folder; see curation/README.md for its contents.
import os, sys
if "data/curation/" not in os.getcwd().replace(os.sep, "/") + "/":
    sys.path.insert(0, os.path.abspath("."))
    os.chdir('../../data/curation/rBMRB')
print("working folder:", os.getcwd())


# Released split

Starts from the old split in `cluster_30p_old/split_output/`, written by `Screening_6_match.ipynb`. All outputs are in `cluster_30p/`.

Homology comes from one exhaustive MMseqs2 all-against-all search over all entries, so that E-values are comparable: identity ≥ 30%, alignment covering ≥ 80% of either sequence, E ≤ 1e-3. The test set is fixed. Validation entries homologous to test, and training entries homologous to test or to the kept validation entries, are removed. So are entries sharing a BMRB accession with those sets.

In [ ]:
import json, shutil, subprocess
from collections import defaultdict
from pathlib import Path
import pandas as pd

OLD_SPLIT_DIR = Path("cluster_30p_old/split_output")
DB_JSON       = Path("bmrb_cs_data_withoffset_lacs_v2.json")   # LACS ∩ new PANAV (6123 entries)
SRC_DIR       = Path("LACS_PANAV_structures_v2")               # file names may carry a test_ prefix
OUT_ROOT      = Path("cluster_30p")
SPLITS        = ("train", "val", "test")

FOLDSEEK = "foldseek"   # on PATH (curation/README.md)
MMSEQS   = "mmseqs"     # on PATH (curation/README.md)

# Leakage criteria (a hit = homologous pair)
MIN_ID     = 0.3    # sequence identity of the alignment
MIN_COV    = 0.8    # coverage of EITHER sequence: max(qcov, tcov)
MAX_EVALUE = 1e-3   # significance, from one all-against-all search over all candidates
FILTER_VAL_TRAIN = True   # True: also remove train entries homologous to retained val entries

def bare(stem):
    """Strip the test_ prefix used in the structure file names."""
    return stem[len("test_"):] if stem.startswith("test_") else stem

def read_ids(path):
    return [l.strip() for l in open(path) if l.strip()]

def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(ids) + "\n")

## 1. Remove entries that left the database

In [ ]:
with open(DB_JSON) as f:
    db_ids = {e.removeprefix("bmr") for e in json.load(f)["entry_ID"].values()}

SPLIT_DIR = OUT_ROOT / "split_output"
ids, removed = {}, {}
for s in SPLITS:
    old = read_ids(OLD_SPLIT_DIR / f"{s}_entry_ids.txt")
    ids[s] = [e for e in old if e in db_ids]
    removed[s] = [e for e in old if e not in db_ids]
    write_ids(SPLIT_DIR / f"{s}_entry_ids.txt", ids[s])
    print(f"{s}: {len(old)} -> {len(ids[s])}  removed: {removed[s]}")

write_ids(SPLIT_DIR / "removed_not_in_db.txt", sorted(sum(removed.values(), [])))
all_split = set().union(*map(set, ids.values()))
print(f"total {len(all_split)} | in DB but not in any split: {len(db_ids - all_split)}")
assert not set(ids["train"]) & set(ids["val"]) and not set(ids["train"]) & set(ids["test"]) and not set(ids["val"]) & set(ids["test"])

## 2. Leakage filter and check

In [ ]:
WORK = OUT_ROOT / "leak_filter"
shutil.rmtree(WORK, ignore_errors=True); WORK.mkdir(parents=True)
FMT = "query,target,fident,alnlen,qcov,tcov,evalue,bits"

src = {bare(p.stem): p for p in SRC_DIR.glob("*.pdb")}
all_ids = [e for s in SPLITS for e in ids[s]]
missing = [e for e in all_ids if e not in src]
assert not missing, f"structures missing for {len(missing)} entries: {missing[:5]}"

def run(*args):
    subprocess.run([str(a) for a in args], check=True, capture_output=True)

# 1. One exhaustive all-against-all sequence search over every candidate entry
d = WORK / "all"; d.mkdir()
for e in all_ids:
    (d / f"{e}.pdb").symlink_to(src[e].resolve())
run(FOLDSEEK, "createdb", d, WORK / "alldb")
run(FOLDSEEK, "convert2fasta", WORK / "alldb", WORK / "all.fa")
m8 = WORK / "all_vs_all.m8"
run(MMSEQS, "easy-search", WORK / "all.fa", WORK / "all.fa", m8, WORK / "tmp_mm",
    "--exhaustive-search", "1", "-e", "10", "--threads", "8", "--max-seqs", "10000",
    "--format-output", FMT)
aln = pd.read_csv(m8, sep="\t", names=FMT.split(","))
for c in ("query", "target"):
    aln[c] = aln[c].astype(str).str.removesuffix(".pdb")

hits = aln[(aln["query"] != aln["target"])
           & (aln.fident >= MIN_ID)
           & (aln[["qcov", "tcov"]].max(axis=1) >= MIN_COV)
           & (aln.evalue <= MAX_EVALUE)]
nbr = defaultdict(set)   # undirected homology graph
for q, t in zip(hits["query"], hits["target"]):
    nbr[q].add(t); nbr[t].add(q)
print(f"{len(aln)} alignments, {len(hits)} pass the criteria")

# 2. Filter (test fixed)
acc = lambda e: e.split("_")[0]   # BMRB accession
test = set(ids["test"])
test_acc = {acc(e) for e in test}
val_keep = [v for v in ids["val"] if not (nbr[v] & test) and acc(v) not in test_acc]
guard = test | set(val_keep) if FILTER_VAL_TRAIN else test
guard_acc = test_acc | {acc(v) for v in val_keep}   # same-accession rule always applies
train_keep = [t for t in ids["train"] if not (nbr[t] & guard) and acc(t) not in guard_acc]

OUT = OUT_ROOT / "split_output_filtered"
shutil.rmtree(OUT, ignore_errors=True)
final = {"train": train_keep, "val": val_keep, "test": ids["test"]}
for s in SPLITS:
    write_ids(OUT / f"{s}_entry_ids.txt", final[s])
write_ids(OUT / "val_dropped.txt", sorted(set(ids["val"]) - set(val_keep)))
write_ids(OUT / "train_dropped.txt", sorted(set(ids["train"]) - set(train_keep)))
split_of = {e: s for s in SPLITS for e in ids[s]}
cross = hits.assign(query_split=hits["query"].map(split_of), target_split=hits["target"].map(split_of))
cross[cross.query_split != cross.target_split].to_csv(OUT / "cross_split_hits_before_filter.tsv", sep="\t", index=False)
print(f"test {len(ids['test'])} (fixed) | val {len(ids['val'])} -> {len(val_keep)} "
      f"| train {len(ids['train'])} -> {len(train_keep)}")

# 3. Verification on the same hit graph: no homologous pair / shared accession across checked boundaries
fin = {e: s for s in SPLITS for e in final[s]}
checked = {("test", "train"), ("test", "val")} | ({("val", "train")} if FILTER_VAL_TRAIN else set())
def n_cross(a, b):
    n_hom = sum(1 for e in final[a] for x in nbr[e] if fin.get(x) == b)
    n_acc = len({acc(e) for e in final[a]} & {acc(e) for e in final[b]})
    return n_hom, n_acc
for a, b in (("test", "train"), ("test", "val"), ("val", "train")):
    n_hom, n_acc = n_cross(a, b)
    tag = "checked" if (a, b) in checked else "not filtered (val is used only for model selection)"
    print(f"{a} vs {b}: {n_hom} homologous pairs, {n_acc} shared accessions  [{tag}]")
    if (a, b) in checked:
        assert n_hom == 0, f"leakage remains between {a} and {b}"
    assert n_acc == 0, f"shared BMRB accession between {a} and {b}"

(OUT / "criteria.json").write_text(json.dumps({
    "homology": {"min_identity": MIN_ID, "min_coverage_either": MIN_COV, "max_evalue": MAX_EVALUE,
                 "search": "MMseqs2 easy-search, exhaustive, all-against-all over all candidate entries"},
    "filter_val_train": FILTER_VAL_TRAIN, "same_bmrb_accession_dropped": True, "test_fixed": True,
    "counts": {s: len(final[s]) for s in SPLITS},
}, indent=2))

## 3. Structures

In [ ]:
STRUCT_OUT = OUT_ROOT / "structures_split_filtered"
for s in SPLITS:
    out_dir = STRUCT_OUT / s
    shutil.rmtree(out_dir, ignore_errors=True); out_dir.mkdir(parents=True)
    for e in final[s]:
        shutil.copy2(src[e], out_dir / f"{e}.pdb")   # bare id, no test_ prefix
    print(f"{s}: {len(final[s])} structures -> {out_dir}")

## 4. PANAV, LACS and NOREF json files

In [ ]:
DATASETS = {
    "PANAV": Path("bmrb_cs_data_withoffset_panav_v2.json"),
    "LACS":  Path("bmrb_cs_data_withoffset_lacs_v2.json"),
    "NOREF": Path("bmrb_cs_data_withoffset_noref_v1.json"),
}

def strip_bmr(entry_id):
    return entry_id[len("bmr"):] if entry_id.startswith("bmr") else entry_id

for model, path in DATASETS.items():
    with open(path) as f:
        data = json.load(f)
    bare_to_key = {strip_bmr(v): k for k, v in data["entry_ID"].items()}
    out_dir = OUT_ROOT / model
    out_dir.mkdir(parents=True, exist_ok=True)
    for s in SPLITS:
        missing = [e for e in final[s] if e not in bare_to_key]
        assert not missing, f"{model}/{s}: {len(missing)} entries missing, e.g. {missing[:5]}"
        keys = {bare_to_key[e] for e in final[s]}
        out = {field: {k: (strip_bmr(v) if field == "entry_ID" else v)
                       for k, v in values.items() if k in keys}
               for field, values in data.items()}
        with open(out_dir / f"{s}.json", "w") as f:
            json.dump(out, f, indent=4)
        print(f"{model} {s}: {len(keys)} entries -> {out_dir / f'{s}.json'}")